In [104]:
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import joblib
import numpy as np

In [105]:
X_user_train = pd.read_csv("../data/X_user_train_scaled.csv")
X_movie_train = pd.read_csv("../data/X_movie_train_scaled.csv")
y_train = pd.read_csv("../data/y_train_scaled.csv")

X_user_test = pd.read_csv("../data/X_user_test_scaled.csv")
X_movie_test = pd.read_csv("../data/X_movie_test_scaled.csv")
y_test = pd.read_csv("../data/y_test_scaled.csv")

scaler_users = joblib.load('../data/scaler_users.pkl')
scaler_movies = joblib.load('../data/scaler_movies.pkl')
scaler_y = joblib.load('../data/scaler_y.pkl')

In [106]:
print("X_user_train shape:", X_user_train.shape)
print("X_movie_train shape:", X_movie_train.shape)
print("y_train shape:", y_train.shape)

X_user_train shape: (149173, 32)
X_movie_train shape: (149173, 594)
y_train shape: (149173, 1)


In [ ]:
#Build initial NN

from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
#poorer results when reg applied, so not used currently

num_outputs = 32

tf.random.set_seed(42)

userNN = tf.keras.models.Sequential([
    Dense(256, activation='relu'),
    Dropout(0.4),

    Dense(128, activation='relu'),
    Dropout(0.4),

    Dense(num_outputs, activation='linear')
])

movieNN = tf.keras.models.Sequential([

    Dense(256, activation='relu'),
    Dropout(0.4),

    Dense(128, activation='relu'),
    Dropout(0.4),

    Dense(num_outputs, activation='linear')
])

#create user input and point to base network
input_user = tf.keras.layers.Input(shape=(X_user_train.shape[1],))
vu = userNN(input_user)
#normalise to unit vector
vu = tf.keras.ops.normalize(vu, axis=1)


#create movie input and point to base network
input_movie = tf.keras.layers.Input(shape=(X_movie_train.shape[1],))
vm = movieNN(input_movie)
#normalise to unit vector
vm = tf.keras.ops.normalize(vm, axis=1)

#dot product layer
output = tf.keras.layers.Dot(axes=1)([vu, vm])

#model spec
model = tf.keras.Model(inputs=[input_user, input_movie], outputs=output)

model.summary()



Model: "functional_80"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_104     │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_106     │ (None, 594)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_52       │ (None, 32)        │     45,472 │ input_layer_104[… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_53       │ (None, 32)        │    189,344 │ input_layer_106[… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_52        │ (None, 32)        │          0 │ sequential_52[0]… │
│ (Normalize)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_53        │ (None, 32)        │          0 │ sequential_53[0]… │
│ (Normalize)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_26 (Dot)        │ (None, 1)         │          0 │ normalize_52[0][… │
│                     │                   │            │ normalize_53[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 234,816 (917.25 KB)

 Trainable params: 234,816 (917.25 KB)

 Non-trainable params: 0 (0.00 B)

output = 32

userNN = 
256 -> 128

movieNN = 
256 -> 128

In [168]:
tf.random.set_seed(42)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,loss=cost_fn)

In [169]:
tf.random.set_seed(42)
model.fit([X_user_train, X_movie_train], y_train, epochs=30)

Epoch 1/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.1590
Epoch 2/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.1494
Epoch 3/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.1457
Epoch 4/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.1433
Epoch 5/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.1414
Epoch 6/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.1402
Epoch 7/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.1387
Epoch 8/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.1382
Epoch 9/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.1372
Epoch 10/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.1361
Epoch 11/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.1355
Epoch 12/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.1348
Epoch 13/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.1338
Epoch 14/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.1337
Epoch 15/30
4662/4662 ━━━

In [170]:
#MSE eval on test set
model.evaluate([X_user_test, X_movie_test], y_test)

#test loss close to training loss, so model is not overfitting
#TEST LOSS - 0.1196

#30 epochs and 1000 training samples
#TRAINING LOSS
#0.15645070374011993 - only genre features
#0.1541982740163803 - genre and decade features
#0.17613600194454193 - movie vector enriched with TMDB features (overfitting, pre dropout)
#0.15593589842319489 - movie vector enriched with TMDB features (overfitting, dropout 0.5)
#0.15448433160781868 - movie vector enriched with TMDB features, decreased dimensionality and more selective threshold (dropout 0.4)

1166/1166 ━━━━━━━━━━━━━━━━━━━━ 1s 884us/step - loss: 0.1549


0.1548721343278885

In [171]:
y_train_pred_scaled = model.predict([X_user_train, X_movie_train]).flatten()
y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled.reshape(-1, 1)).flatten()

y_test_pred_scaled = model.predict([X_user_test, X_movie_test]).flatten()
y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).flatten()

#target y values
y_train_unscaled = scaler_y.inverse_transform(y_train).flatten()
y_test_unscaled = scaler_y.inverse_transform(y_test).flatten()

4662/4662 ━━━━━━━━━━━━━━━━━━━━ 4s 911us/step
1166/1166 ━━━━━━━━━━━━━━━━━━━━ 1s 920us/step


In [172]:
#ADD RMSE, MAE, R2
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Evaluation function
def evaluate_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return rmse, mae, r2

trainig_rmse, training_mae, training_r2 = evaluate_model(y_train_unscaled, y_train_pred)

print(f"Training RMSE:   {trainig_rmse:.4f}")
print(f"Training MAE:    {training_mae:.4f}")
print(f"Training R²:     {training_r2:.4f}")

print()

test_rmse, test_mae, test_r2 = evaluate_model(y_test_unscaled, y_test_pred)

print(f"Test RMSE:       {test_rmse:.4f}")
print(f"Test MAE:        {test_mae:.4f}")
print(f"Test R²:         {test_r2:.4f}")



Training RMSE:   0.7700
Training MAE:    0.5880
Training R²:     0.4777

Test RMSE:       0.8846
Test MAE:        0.6686
Test R²:         0.3138
